In [1]:

import os
os.environ.pop('MPLBACKEND', None)

os.environ['MPLBACKEND'] = 'agg'

import matplotlib.pyplot as plt
import seaborn as sns


# ==============================================================================
# 0. PEFT IMPORTS AND INITIAL SETUP (NEW)
# ==============================================================================
# Ensure you have the peft library installed: pip install peft
from peft import LoraConfig, get_peft_model, TaskType

print("\n" + "="*80)
print("... 0. PEFT LIBRARY INITIALIZED ...")
print("    Ready for Parameter-Efficient Fine-Tuning with LoRA!")
# ==============================================================================
# 1. IMPORTS AND INITIAL SETUP
# ==============================================================================
import os
import sys
import time
import json
import random
import pickle
from pathlib import Path
from datetime import datetime

# Standard data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# In the IMPORTS AND INITIAL SETUP section
from tqdm import tqdm
import warnings

# Machine learning libraries
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from sklearn.cluster import AgglomerativeClustering


# PyTorch and Transformers
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer

# Advanced CV imports
from Bio.Align import PairwiseAligner
from scipy.cluster.hierarchy import linkage, fcluster
from multiprocessing import Pool, cpu_count
from typing import List, Tuple

# Set style for plots


# --- MODIFIED: Title reflects the use of ESM-2 ---
print("🧬 BALM Adaptation for ESM-2 with LoRA Fine-Tuning")
print("⚡ Fast Training with 5-Fold Cross-Validation")
print("="*80)

def setup_reproducibility(seed=42):
    """Setup complete reproducibility."""
    print(f"🔒 Setting up reproducibility with seed {seed}")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"✅ Reproducibility setup complete")

setup_reproducibility(42)

# ==============================================================================
# 2. ESM-2 EMBEDDING EXTRACTOR (Modified for LoRA)
# ==============================================================================

class ProteinEmbeddingExtractor:
    """
    A class to prepare the ESM-2 model, optionally with LoRA adapters.
    This class handles model loading and tokenization.
    """
    def __init__(
        self,
        model_name="facebook/esm2_t33_650M_UR50D",
        device="auto",
        lora_rank=8,
        lora_alpha=16,
        lora_dropout=0.1,
        use_lora=True
    ):
        self.model_name = model_name
        self.use_lora = use_lora
        
        if device == "auto":
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)
            
        print(f"🔧 Initializing ProteinEmbeddingExtractor on {self.device}")
        
        self.dtype = torch.float16 if self.device.type == 'cuda' else torch.float32
        print(f"📥 Loading ESM-2 model: {model_name} with dtype: {self.dtype}")
        
        self.model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=self.dtype
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        if self.use_lora:
            print(f"✨ Applying LoRA with rank={lora_rank}, alpha={lora_alpha}, dropout={lora_dropout}")
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                bias="none",
                task_type=TaskType.FEATURE_EXTRACTION,
                target_modules=["key", "query", "value"]
            )
            self.model = get_peft_model(self.model, lora_config)
            print("   ✅ LoRA model loaded and configured. Trainable parameters:")
            self.model.print_trainable_parameters()
        else:
            print("   ⚠️ LoRA is disabled. All base model weights are frozen.")
            for param in self.model.parameters():
                param.requires_grad = False

        self.model.to(self.device)
        self.embedding_size = self.model.config.hidden_size
        print(f"   ✅ Model loaded. Embedding size from config: {self.embedding_size}")

    def get_model_and_tokenizer(self):
        """Returns the prepared model (PeftModel or base) and tokenizer."""
        return self.model, self.tokenizer

# ==============================================================================
# 3. LOAD YOUR DATASET
# ==============================================================================
print("\n" + "="*80)
print("... 3. LOADING DATA FROM Data.csv ...")

protein1_col = 'Target'
protein2_col = 'proteina'
target_col = 'Y'
pdb_col_raw = 'PDB'
subgroup_col = 'Subgroup'
source_dataset_col = 'Source Data Set' # NEW
try:
    df_raw = pd.read_csv(r"C:\Users\hs494\PPIBALM\BALM-PPI\scripts\notebooks\PPB_Affinity_Sequences_Final (version 1).csv")
    print(f"Successfully loaded Data.csv with {len(df_raw)} rows.")
    df = df_raw[[protein1_col, protein2_col, target_col, pdb_col_raw, subgroup_col,source_dataset_col]].copy()
    df.rename(columns={
        protein1_col: 'Target',
        protein2_col: 'proteina',
        target_col: 'Y',
        subgroup_col : 'Subgroup',
        source_dataset_col: 'SourceDataSet' # NEW
    }, inplace=True)
    
    print("\n--- Initial Data Check ---")
    print(df.head())
    missing_values = df.isnull().sum()
    print("\nMissing values per column:\n", missing_values)
    
    if missing_values.sum() > 0:
        print(f"\nWarning: Found {missing_values.sum()} total missing values. Dropping rows with NaNs.")
        df.dropna(subset=['Target', 'proteina', 'Y', 'PDB'], inplace=True)
        print(f"Dataset now has {len(df)} rows after cleaning.")
        
    df['Target'] = df['Target'].astype(str)
    df['proteina'] = df['proteina'].astype(str)
    df['Y'] = pd.to_numeric(df['Y'], errors='coerce')
    df['PDB'] = df['PDB'].astype(str)
    df['Subgroup'] = df['Subgroup'].astype(str)
    df['SourceDataSet'] = df['SourceDataSet'].astype(str) # NEW

    df.dropna(subset=['Y'], inplace=True)
    
    print(f"\n✅ Data is loaded and cleaned. Final dataset size: {len(df)} samples.")
    df.info()

except FileNotFoundError:
    print("❌ ERROR: 'Data.csv' not found.")
    df = pd.DataFrame(columns=['Target', 'proteina', 'Y', 'PDB'])
except KeyError as e:
    print(f"❌ ERROR: Column {e} not found in 'Data.csv'.")
    df = pd.DataFrame(columns=['Target', 'proteina', 'Y', 'PDB'])

# ==============================================================================
# 4. EXPERIMENT CONFIGURATION
# ==============================================================================
print("\n" + "="*80)
print("... 4. CONFIGURING THE EXPERIMENT ...")

CV_STRATEGY = 'SequenceSimilarity' 
N_SPLITS = 5
EPOCHS = 30
PATIENCE = 15
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
PROJECTED_SIZE = 256
DROPOUT = 0.1

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
USE_LORA_FOR_TRAINING = True

# Sequence Similarity CV Configs
SIMILARITY_THRESHOLD = 0.3
K_MER_SIZE = 3
CLUSTERING_METHOD = 'agglomerative'

RESULTS_DIR = "cv_results_lora_seqsim" # Use a unique name
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved in: {RESULTS_DIR}")

print(f"✅ Configuration set. Using CV Strategy: {CV_STRATEGY}")
print(f"   Epochs: {EPOCHS}, Patience: {PATIENCE}")
if USE_LORA_FOR_TRAINING:
    print(f"   LoRA Enabled: Rank={LORA_RANK}, Alpha={LORA_ALPHA}, Dropout={LORA_DROPOUT}")
else:
    print(f"   LoRA Disabled: Only projection head will be trained.")

# ==============================================================================
# 5. ADAPTED MODEL AND DATASET
# ==============================================================================
print("\n" + "="*80)
print("... 5. DEFINING ADAPTED MODEL AND DATASET ...")

class BALMProjectionHead(nn.Module):
    def __init__(self, embedding_size, projected_size, projected_dropout):
        super().__init__()
        self.protein_projection = nn.Linear(embedding_size, projected_size)
        self.proteina_projection = nn.Linear(embedding_size, projected_size)
        self.dropout = nn.Dropout(projected_dropout)
        self.loss_fn = nn.MSELoss()
        print(f"✅ BALMProjectionHead initialized with embedding_size: {embedding_size}")

    def forward(self, protein_embedding, proteina_embedding, labels=None):
        protein_embedding = self.dropout(protein_embedding)
        proteina_embedding = self.dropout(proteina_embedding)
        protein_projected = self.protein_projection(protein_embedding)
        proteina_projected = self.proteina_projection(proteina_embedding)
        protein_projected = F.normalize(protein_projected, p=2, dim=1)
        proteina_projected = F.normalize(proteina_projected, p=2, dim=1)
        cosine_similarity = F.cosine_similarity(protein_projected, proteina_projected)
        cosine_similarity = torch.clamp(cosine_similarity, -0.9999, 0.9999)
        output = {"cosine_similarity": cosine_similarity}
        if labels is not None:
            loss = self.loss_fn(cosine_similarity, labels)
            output["loss"] = loss
        return output

class BALMForLoRAFinetuning(nn.Module):
    def __init__(self, esm_model, esm_tokenizer, projected_size, projected_dropout, pkd_bounds):
        super().__init__()
        self.esm_model = esm_model
        self.esm_tokenizer = esm_tokenizer
        self.embedding_size = self.esm_model.config.hidden_size
        self.projection_head = BALMProjectionHead(self.embedding_size, projected_size, projected_dropout)
        self.pkd_lower, self.pkd_upper = pkd_bounds
        self.pkd_range = self.pkd_upper - self.pkd_lower
        print(f"✅ BALMForLoRAFinetuning model initialized.")

    def _get_esm_embeddings(self, sequences: List[str]):
        cls_token = self.esm_tokenizer.cls_token
        processed_seqs = [s.replace('|', f"{cls_token}{cls_token}") for s in sequences]
        
        inputs = self.esm_tokenizer(processed_seqs, return_tensors="pt", padding=True, truncation=True, max_length=1024)
        inputs = {k: v.to(self.esm_model.device) for k, v in inputs.items()}
        outputs = self.esm_model(**inputs)
        
        # Pooling logic (remains Mean Pooling, which now covers the internal CLS tokens)
        hidden_states = outputs.last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).expand(hidden_states.size()).float()
        return torch.sum(hidden_states * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

    def forward(self, batch_input):
        protein_emb = self._get_esm_embeddings(batch_input["protein_sequence"])
        proteina_emb = self._get_esm_embeddings(batch_input["proteina_sequence"])
        labels = batch_input.get("labels")
        proj_output = self.projection_head(protein_emb, proteina_emb, labels)
        return {
            "cosine_similarity": proj_output["cosine_similarity"],
            "original_pkds": batch_input["original_pkds"],
            "pdb_groups": batch_input["pdb_groups"],
            "subgroups": batch_input["subgroups"],
            "source_dataset": batch_input["source_dataset"], # NEW
            "loss": proj_output.get("loss")
        }
        if "loss" in proj_output:

            output["loss"] = proj_output["loss"]
        return output

class ProteinPairSequenceDataset(Dataset):
    def __init__(self, dataframe, pkd_bounds):
        self.data = dataframe.reset_index(drop=True)
        self.pkd_lower, self.pkd_upper = pkd_bounds
        self.pkd_range = self.pkd_upper - self.pkd_lower

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        original_pkd = float(row['Y'])
        scaled_label = ((original_pkd - self.pkd_lower) / self.pkd_range) * 2 - 1
        return {
            "protein_sequence": row["Target"],
            "proteina_sequence": row["proteina"],
            "labels": torch.tensor(scaled_label, dtype=torch.float32),
            "original_pkds": torch.tensor(original_pkd, dtype=torch.float32),
            "pdb_groups": row["PDB"],
            "subgroups": row["Subgroup"],
            "source_dataset": row["SourceDataSet"] # NEW
        }

def collate_fn_sequences(batch):
    return {
        "protein_sequence": [item['protein_sequence'] for item in batch],
        "proteina_sequence": [item['proteina_sequence'] for item in batch],
        "labels": torch.stack([item['labels'] for item in batch]),
        "original_pkds": torch.stack([item['original_pkds'] for item in batch]),
        "pdb_groups": [item['pdb_groups'] for item in batch], # NEW
        "subgroups": [item['subgroups'] for item in batch],  # NEW
        "source_dataset": [item['source_dataset'] for item in batch] # NEW
    }

# ==============================================================================
# 6. METRICS, EVALUATION & SEQUENCE SIMILARITY HELPERS (NEW)
# ==============================================================================
print("\n" + "="*80)
print("... 6. DEFINING METRICS, EVALUATION & SEQUENCE HELPERS ...")

def concordance_index(y_true, y_pred):
    try:
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        n = len(y_true)
        if n < 2: return 1.0
        concordant, discordant, tied_pred = 0, 0, 0
        for i in range(n):
            for j in range(i + 1, n):
                if y_true[i] == y_true[j]: continue
                if y_pred[i] == y_pred[j]:
                    tied_pred += 1
                    continue
                if (y_pred[i] > y_pred[j] and y_true[i] > y_true[j]) or \
                   (y_pred[i] < y_pred[j] and y_true[i] < y_true[j]):
                    concordant += 1
                else:
                    discordant += 1
        total_pairs = concordant + discordant + tied_pred
        return 1.0 if total_pairs == 0 else (concordant + 0.5 * tied_pred) / total_pairs
    except Exception:
        return np.nan

def evaluate_model(model, data_loader, device, pkd_bounds):
    model.eval()
    all_labels, all_preds_pkd, all_pdb_groups, all_subgroups, all_sources = [], [], [], [], [] 
    pkd_lower, pkd_range = pkd_bounds[0], pkd_bounds[1] - pkd_bounds[0]
    
    with torch.no_grad():
        for batch in data_loader:
            batch_gpu = {k: v for k, v in batch.items()}
            for key in ["labels", "original_pkds"]:
                if key in batch_gpu: batch_gpu[key] = batch_gpu[key].to(device)
            outputs = model(batch_gpu)
            cosine_sim = outputs['cosine_similarity'].cpu()
            preds_pkd = ((cosine_sim + 1) / 2) * pkd_range + pkd_lower
            all_labels.extend(batch['original_pkds'].numpy())
            all_preds_pkd.extend(preds_pkd.numpy())
            all_pdb_groups.extend(batch['pdb_groups']) 
            all_subgroups.extend(batch['subgroups'])   
            all_sources.extend(batch['source_dataset']) # NEW

    labels, preds = np.array(all_labels), np.array(all_preds_pkd)
    pdb_groups_arr = np.array(all_pdb_groups)
    subgroups_arr = np.array(all_subgroups)
    sources_arr = np.array(all_sources) # NEW
    
    mask = ~np.isnan(labels) & ~np.isnan(preds)
    labels, preds = labels[mask], preds[mask]
    
    if len(labels) < 2:
        return {"rmse": np.nan, "pearson": np.nan, "spearman": np.nan, "ci": np.nan}, labels, preds, pdb_groups_arr, subgroups_arr, sources_arr
    return {
        'rmse': np.sqrt(mean_squared_error(labels, preds)),
        'pearson': pearsonr(labels, preds)[0],
        'spearman': spearmanr(labels, preds)[0],
        'ci': concordance_index(labels, preds)
    }, labels, preds, pdb_groups_arr, subgroups_arr, sources_arr

def plot_regression(y_true, y_pred, metrics, title, filename=None):
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.6, edgecolor='k', s=50)
    min_val, max_val = min(np.min(y_true), np.min(y_pred)), max(np.max(y_true), np.max(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Identity Line (y=x)')
    metrics_text = f"RMSE: {metrics['rmse']:.4f}\nPearson: {metrics['pearson']:.4f}\nSpearman: {metrics['spearman']:.4f}\nCI: {metrics['ci']:.4f}"
    plt.text(0.05, 0.95, metrics_text, transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5))
    plt.title(title, fontsize=16)
    plt.xlabel("Actual pKd", fontsize=14)
    plt.ylabel("Predicted pKd", fontsize=14)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if filename:                       # NEW
        plt.savefig(filename)          # NEW
        print(f"Plot saved to {filename}") # NEW
    plt.show()

def compute_kmer_similarity_matrix(sequences, k=3):
    """
    Compute pairwise sequence similarity matrix using k-mer based Jaccard similarity.
    This is a biologically relevant method for sequence similarity computation.
    """
    print(f"Computing {k}-mer Jaccard similarity matrix for {len(sequences)} sequences...")
    
    def get_kmers(sequence, k_size):
        """Extract k-mers from a sequence."""
        if len(sequence) < k_size:
            return {sequence}
        return {sequence[i:i+k_size] for i in range(len(sequence) - k_size + 1)}
    
    def jaccard_similarity(seq1, seq2):
        """Compute Jaccard similarity between two sequences based on k-mers."""
        kmers1 = get_kmers(seq1, k)
        kmers2 = get_kmers(seq2, k)
        if not kmers1 or not kmers2:
            return 0.0
        intersection = len(kmers1 & kmers2)
        union = len(kmers1 | kmers2)
        return intersection / union if union > 0 else 0.0
    
    n_seqs = len(sequences)
    similarity_matrix = np.zeros((n_seqs, n_seqs))
    
    # Compute similarity matrix with progress bar
    for i in tqdm(range(n_seqs), desc="Computing similarities"):
        for j in range(i, n_seqs):
            if i == j:
                similarity_matrix[i, j] = 1.0
            else:
                sim = jaccard_similarity(sequences[i], sequences[j])
                similarity_matrix[i, j] = sim
                similarity_matrix[j, i] = sim
    
    avg_similarity = np.mean(similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)])
    print(f"Average pairwise similarity: {avg_similarity:.3f}")
    
    return similarity_matrix

def cluster_sequences_by_similarity(similarity_matrix, threshold=0.3, method='agglomerative'):
    """
    Cluster sequences based on similarity matrix using specified clustering method.
    Uses distance threshold derived from similarity threshold.
    """
    print(f"Clustering sequences using {method} method with similarity threshold {threshold}")
    
    # Convert similarity to distance matrix
    distance_matrix = 1.0 - similarity_matrix
    
    try:
        if method == 'agglomerative':
            # Use distance threshold - sequences with similarity > threshold will be in same cluster
            distance_threshold = 1.0 - threshold
            
            print(f"Using distance threshold: {distance_threshold}")
            print(f"Distance matrix stats - Min: {distance_matrix.min():.3f}, Max: {distance_matrix.max():.3f}, Mean: {distance_matrix.mean():.3f}")
            
            clustering = AgglomerativeClustering(
                n_clusters=None,
                distance_threshold=distance_threshold,
                linkage='average',  # Average linkage works well for biological sequences
                metric='precomputed'
            )
            
            clusters = clustering.fit_predict(distance_matrix)
            print(f"AgglomerativeClustering completed successfully")
            
        else:
            raise ValueError(f"Unsupported clustering method: {method}")
        
        if clusters is None:
            raise ValueError("Clustering failed - returned None")
        
        n_clusters = len(set(clusters))
        print(f"Created {n_clusters} clusters")
        
        cluster_sizes = [np.sum(clusters == i) for i in set(clusters)]
        print(f"Cluster size distribution - Min: {min(cluster_sizes)}, Max: {max(cluster_sizes)}, Mean: {np.mean(cluster_sizes):.1f}")
        
        return clusters
        
    except Exception as e:
        print(f"❌ Clustering failed with error: {e}")
        print("🔄 Falling back to individual clusters (each sequence in its own cluster)")
        
        n_sequences = similarity_matrix.shape[0]
        fallback_clusters = np.arange(n_sequences)
        
        print(f"Created {n_sequences} individual clusters (fallback)")
        return fallback_clusters

# ==============================================================================
# 7. CROSS-VALIDATION TRAINING (MODIFIED FOR 80/20 SPLIT + TRAIN LOSS PATIENCE)
# ==============================================================================
print("\n" + "="*80)
print(f"... 7. RUNNING '{CV_STRATEGY}' STRATEGY WITH LoRA (80/20 Split with Training Loss Patience) ...")

def run_sequence_similarity_cv_with_lora():
    if df.empty:
        print("⚠️ Skipping training due to empty dataframe.")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    PKD_LOWER_BOUND, PKD_UPPER_BOUND = df['Y'].min(), df['Y'].max()
    PKD_BOUNDS = (PKD_LOWER_BOUND, PKD_UPPER_BOUND)
    print(f"Global pKd bounds for scaling: [{PKD_LOWER_BOUND:.2f}, {PKD_UPPER_BOUND:.2f}]")

    # --- Steps 1-4: Data Preparation and Clustering ---
    print("\n🧬 Step 1: Extracting unique protein sequences...")
    sequence_columns = ['Target', 'proteina']
    all_sequences = set()
    sequence_to_records = {}
    for idx, row in df.iterrows():
        for col in sequence_columns:
            seq = str(row[col]).strip()
            if seq:
                all_sequences.add(seq)
                if seq not in sequence_to_records:
                    sequence_to_records[seq] = []
                sequence_to_records[seq].append(idx)
    unique_sequences = sorted(list(all_sequences))
    print(f"Found {len(unique_sequences)} unique protein sequences.")

    print("\n🧬 Step 2: Computing sequence similarity matrix...")
    similarity_matrix = compute_kmer_similarity_matrix(unique_sequences, k=K_MER_SIZE)

    print("\n🧬 Step 3: Clustering sequences...")
    clusters = cluster_sequences_by_similarity(similarity_matrix, threshold=SIMILARITY_THRESHOLD, method=CLUSTERING_METHOD)
    
    cluster_to_records = {}
    for seq_idx, cluster_id in enumerate(clusters):
        seq = unique_sequences[seq_idx]
        if cluster_id not in cluster_to_records:
            cluster_to_records[cluster_id] = []
        cluster_to_records[cluster_id].extend(sequence_to_records[seq])
    
    for cluster_id in cluster_to_records:
        cluster_to_records[cluster_id] = list(set(cluster_to_records[cluster_id]))
    # --- End of omitted code ---


    # --- Step 5: Perform Cluster-based Cross-Validation ---
    print(f"\n🚀 Starting {N_SPLITS}-Fold **{CV_STRATEGY}** Cross-Validation with LoRA fine-tuning...")
    
    # NOTE: Model initialization moved INSIDE the loop to prevent weight leakage.
    
    cluster_ids = list(cluster_to_records.keys())
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    
    fold_results, all_folds_y_true, all_folds_y_pred = [], [], []
    
    for fold, (train_cluster_idx, test_cluster_idx) in enumerate(kf.split(cluster_ids)):
        fold_num = fold + 1
        print(f"\n===== STRATEGY: {CV_STRATEGY} | FOLD {fold_num}/{N_SPLITS} =====")
        
        # --- CRITICAL FIX: RE-INITIALIZE MODEL HERE ---
        # This ensures the LoRA weights are reset for every fold.
        print(f"🔄 Initializing fresh ESM-2 Model and LoRA Adapters for Fold {fold_num}...")
        extractor = ProteinEmbeddingExtractor(
            device=device, use_lora=USE_LORA_FOR_TRAINING, lora_rank=LORA_RANK,
            lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT
        )
        esm_model, esm_tokenizer = extractor.get_model_and_tokenizer()
        # ----------------------------------------------

        fold_model_path = os.path.join(RESULTS_DIR, f"best_model_fold_{fold_num}.pth")
        
        train_clusters = [cluster_ids[i] for i in train_cluster_idx]
        test_clusters = [cluster_ids[i] for i in test_cluster_idx]

        # Create a map from each unique sequence to its group ('train' or 'test')
        sequence_to_group = {}
        train_cluster_set = set(train_clusters)
        test_cluster_set = set(test_clusters)

        for seq_idx, cluster_id in enumerate(clusters):
            sequence = unique_sequences[seq_idx]
            if cluster_id in train_cluster_set:
                sequence_to_group[sequence] = 'train'
            elif cluster_id in test_cluster_set:
                sequence_to_group[sequence] = 'test'

        # Iterate through the main dataframe and assign rows
        train_indices, test_indices = [], []
        
        for idx, row in df.iterrows():
            seq_target = str(row['Target']).strip()
            seq_proteina = str(row['proteina']).strip()

            cluster_target_id = sequence_to_group.get(seq_target)
            cluster_proteina_id = sequence_to_group.get(seq_proteina)

            # Strict Leakage Rule: If EITHER protein is in a test cluster, it goes to test.
            if cluster_target_id == 'test' or cluster_proteina_id == 'test':
                test_indices.append(idx)
            else:
                train_indices.append(idx)

        # Create dataframes
        train_df = df.loc[train_indices]
        test_df = df.loc[test_indices]
        
        assert len(set(train_indices).intersection(set(test_indices))) == 0, "Overlap found between train and test sets!"

        print(f"  Total samples allocated: {len(train_df) + len(test_df)}")
        print(f"  Train: {len(train_df)} samples")
        print(f"  Test: {len(test_df)} samples")

        # --- FIX: Calculate Scaling Bounds using ONLY Training Data (Prevents Data Sniffing) ---
        fold_pkd_min = train_df['Y'].min()
        fold_pkd_max = train_df['Y'].max()
        FOLD_PKD_BOUNDS = (fold_pkd_min, fold_pkd_max)
        print(f"  Fold Scaling Bounds (derived from train set): {FOLD_PKD_BOUNDS}")

        # --- DataLoaders ---
        train_dataset = ProteinPairSequenceDataset(train_df, FOLD_PKD_BOUNDS)
        test_dataset = ProteinPairSequenceDataset(test_df, FOLD_PKD_BOUNDS)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_sequences)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn_sequences)

        # --- Model, Optimizer ---
        model = BALMForLoRAFinetuning(
            esm_model=esm_model, esm_tokenizer=esm_tokenizer,
            projected_size=PROJECTED_SIZE, projected_dropout=DROPOUT,
            pkd_bounds=FOLD_PKD_BOUNDS
        ).to(device)
        
        optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
        print(f"   Optimizing {sum(p.numel() for p in model.parameters() if p.requires_grad)} trainable parameters.")

        # --- Training Loop ---
        best_train_loss = float('inf')
        patience_counter = 0

        for epoch in range(EPOCHS):
            model.train()
            total_train_loss = 0.0
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
            
            for batch in progress_bar:
                optimizer.zero_grad()
                batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
                outputs = model(batch_gpu)
                loss = outputs['loss']
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()
                progress_bar.set_postfix({'batch_loss': f"{loss.item():.4f}"})
            
            avg_train_loss = total_train_loss / len(train_loader)
            # Evaluate on test set (for monitoring only, not for gradient updates)
            val_metrics, _, _, _, _, _ = evaluate_model(model, test_loader, device, FOLD_PKD_BOUNDS)
            print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Test RMSE: {val_metrics['rmse']:.4f}")
            
            # Save best model based on Training Loss (to avoid validation leakage in parameter selection)
            if avg_train_loss < best_train_loss:
                best_train_loss = avg_train_loss
                patience_counter = 0
                torch.save(model.state_dict(), fold_model_path)
                # print(f"   -> New best model saved with Train Loss: {best_train_loss:.4f}")
            else:
                patience_counter += 1
            
            if patience_counter >= PATIENCE:
                print(f"⏳ Early stopping triggered based on training loss at epoch {epoch+1}.")
                break
        
        print(f"Fold {fold_num} training complete.")        
        # --- Final Evaluation on Test Set ---
        print("Loading best model (based on lowest train loss) for final TEST evaluation...")
        model.load_state_dict(torch.load(fold_model_path)) 
        
        final_test_metrics, y_true_fold, y_pred_fold, pdb_groups_fold, subgroups_fold, sources_fold = evaluate_model(model, test_loader, device, PKD_BOUNDS)
        print(f"Fold {fold_num} Final Test Metrics:")
        for key, value in final_test_metrics.items():
            print(f"  - {key.upper()}: {value:.4f}")

        fold_results.append(final_test_metrics)
        all_folds_y_true.extend(y_true_fold)
        all_folds_y_pred.extend(y_pred_fold)

        # --- Save Predictions for the Fold ---
        # MODIFIED: Create a more detailed DataFrame
        pred_df = pd.DataFrame({
            'label': y_true_fold, 
            'prediction': y_pred_fold, 
            'residual': y_true_fold - y_pred_fold,
            'abs_residual': np.abs(y_true_fold - y_pred_fold),
            'PDB': pdb_groups_fold, 
            'Subgroup': subgroups_fold,
            'Source Data Set': sources_fold  # Preserved column
        })
        # MODIFIED: Save to the RESULTS_DIR
        pred_filename = os.path.join(RESULTS_DIR, f"fold_{fold_num}_predictions.csv")
        pred_df.to_csv(pred_filename, index=False)
        print(f"   💾 Predictions for fold {fold_num} saved to '{pred_filename}'")


    # --- Aggregate and Display Final Results  ---
    print(f"\n📈 Generating combined regression plot for {CV_STRATEGY}")
    combined_y_true = np.array(all_folds_y_true)
    combined_y_pred = np.array(all_folds_y_pred)

    overall_metrics = {
        'rmse': np.sqrt(mean_squared_error(combined_y_true, combined_y_pred)),
        'pearson': pearsonr(combined_y_true, combined_y_pred)[0],
        'spearman': spearmanr(combined_y_true, combined_y_pred)[0],
        'ci': concordance_index(combined_y_true, combined_y_pred)
    }
    
    plot_regression(
    combined_y_true,
    combined_y_pred,
    overall_metrics,
    title=f"Overall Regression for {CV_STRATEGY} ({N_SPLITS}-Fold CV) with LoRA Fine-tuning",
    filename=os.path.join(RESULTS_DIR, "overall_regression_plot.png")
    )

    print(f"\n✅ Final Results Summary for {CV_STRATEGY} with LoRA Fine-tuning:")
    results_df = pd.DataFrame(fold_results)
    summary_df = pd.DataFrame({'Mean': results_df.mean(), 'Std Dev': results_df.std()})
    print(summary_df)

    # NEW: Save the summary metrics to a CSV file
    summary_filename = os.path.join(RESULTS_DIR, "cv_summary_metrics.csv")
    summary_df.to_csv(summary_filename)
    print(f"Overall CV summary metrics saved to {summary_filename}")
    print(f"Fold model checkpoints and prediction files are saved in the '{RESULTS_DIR}' directory.")

# --- Call the new CV function ---
run_sequence_similarity_cv_with_lora()


# ==============================================================================
# 8. FINAL NOTES AND RECOMMENDATIONS
# ==============================================================================
print("\n" + "="*80)
print("... 8. FINAL NOTES AND RECOMMENDATIONS ...")
print("""
🎉 CONGRATULATIONS! You've successfully implemented BALM with on-the-fly ESM-2 embeddings
and **Parameter-Efficient Fine-Tuning (PEFT) using LoRA**!

📊 KEY ADVANTAGES OF THIS APPROACH:
• **Efficiency:** Fine-tunes only a tiny fraction of the ESM-2 model parameters (<1%).
• **Speed & Memory:** Significantly faster and less memory-intensive than full fine-tuning.
• **Performance:** Adapts the powerful ESM-2 model to your specific data, often leading to better performance.
• **No Static Cache:** By computing embeddings dynamically, the model fully leverages the fine-tuned LoRA adapters during training.

🚀 PERFORMANCE OPTIMIZATION TIPS:
1.  **LoRA Hyperparameters:**
    *   `lora_rank (r)`: Controls the capacity of the adapters. Higher `r` means more parameters. Typical values are 8-64.
    *   `lora_alpha`: Acts as a scaling factor. A common heuristic is to set `lora_alpha` to be twice the `lora_rank`. [7]
    *   `target_modules`: Experiment with adding other modules like `dense` to the target list in the `LoraConfig`. [1]
2.  **Learning Rate:** LoRA can sometimes handle slightly higher learning rates than full fine-tuning. Experiment with values from 5e-5 to 5e-4.
3.  **Batch Size & Gradient Accumulation:** If your GPU memory is a constraint, you can simulate a larger batch size by using gradient accumulation.

💾 SAVING AND LOADING THE FINE-TUNED MODEL:
• After training, you can save the trained LoRA adapters separately from the base model. This is very efficient for storage. [3]
• To save: `model.esm_model.save_pretrained("path/to/my_lora_adapters")` [3]
• To load: First, load the base ESM-2 model. Then, use `PeftModel.from_pretrained(base_model, "path/to/my_lora_adapters")`.

🎯 NEXT STEPS:
1.  **Hyperparameter Search:** Systematically tune LoRA parameters (`r`, `lora_alpha`) and the learning rate for optimal performance.
2.  **Experiment with Larger ESM-2 Models:** LoRA makes fine-tuning even the largest ESM-2 models (e.g., 3B parameters) feasible on a single high-end GPU.
3.  **Advanced PEFT Techniques:** Explore other PEFT methods like IA3 or full parameter-efficient fine-tuning for comparison.
""")
print("\n✅ LoRA PEFT implementation complete! Happy parameter-efficient protein modeling! 🧬🤖")


... 0. PEFT LIBRARY INITIALIZED ...
    Ready for Parameter-Efficient Fine-Tuning with LoRA!
🧬 BALM Adaptation for ESM-2 with LoRA Fine-Tuning
⚡ Fast Training with 5-Fold Cross-Validation
🔒 Setting up reproducibility with seed 42
✅ Reproducibility setup complete

... 3. LOADING DATA FROM Data.csv ...
Successfully loaded Data.csv with 12062 rows.

--- Initial Data Check ---
                                              Target  \
0  PKFTKCRSPERETFSCHWTLGPIQLFYTRRNTQEWTQEWKECPDYV...   
1  QDNSRYTHFLTQHYDAKPQGRDDRYCESIMRRRGLTSPCKDINTFI...   
2  KSFPEVVGKTVDQAREYFTLHYPQYDVYFLPEGSPVTLDLRYNRVR...   
3  TNTVAAYNLTWKSTNFKTILEWEPKPVNQVYTVQISTKSGDWKSKC...   
4  PIVQNLQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEG...   

                                            proteina          Y   PDB  \
0  FPTIPLSRLFDNAMLRAHRLHQLAFDTYQEFEEAYIPKEQKYSFLQ...   9.045757  1A22   
1  SLDIQSLDIQCEELSDARWAELLPLLQQCQVVRLDDCGLTEARCKD...  15.301030  1A4Y   
2  CGVPAIQPVLSGLIVNGEEAVPGSWPWQVSLQDKTGFHFCGGSLIN...  11.826814  1A

Computing similarities:   0%|          | 13/11598 [00:08<1:59:00,  1.62it/s]


KeyboardInterrupt: 

In [2]:

import os
os.environ.pop('MPLBACKEND', None)

os.environ['MPLBACKEND'] = 'agg'

import matplotlib.pyplot as plt
import seaborn as sns


# ==============================================================================
# 0. PEFT IMPORTS AND INITIAL SETUP (NEW)
# ==============================================================================
# Ensure you have the peft library installed: pip install peft
from peft import LoraConfig, get_peft_model, TaskType

print("\n" + "="*80)
print("... 0. PEFT LIBRARY INITIALIZED ...")
print("    Ready for Parameter-Efficient Fine-Tuning with LoRA!")
# ==============================================================================
# 1. IMPORTS AND INITIAL SETUP
# ==============================================================================
import os
import sys
import time
import json
import random
import pickle
from pathlib import Path
from datetime import datetime

# Standard data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# In the IMPORTS AND INITIAL SETUP section
from tqdm import tqdm
import warnings

# Machine learning libraries
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from sklearn.cluster import AgglomerativeClustering


# PyTorch and Transformers
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer

# Advanced CV imports
from Bio.Align import PairwiseAligner
from scipy.cluster.hierarchy import linkage, fcluster
from multiprocessing import Pool, cpu_count
from typing import List, Tuple

# Set style for plots


# --- MODIFIED: Title reflects the use of ESM-2 ---
print("🧬 BALM Adaptation for ESM-2 with LoRA Fine-Tuning")
print("⚡ Fast Training with 5-Fold Cross-Validation")
print("="*80)

def setup_reproducibility(seed=42):
    """Setup complete reproducibility."""
    print(f"🔒 Setting up reproducibility with seed {seed}")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"✅ Reproducibility setup complete")

setup_reproducibility(42)

# ==============================================================================
# 2. ESM-2 EMBEDDING EXTRACTOR (Modified for LoRA)
# ==============================================================================

class ProteinEmbeddingExtractor:
    """
    A class to prepare the ESM-2 model, optionally with LoRA adapters.
    This class handles model loading and tokenization.
    """
    def __init__(
        self,
        model_name="facebook/esm2_t33_650M_UR50D",
        device="auto",
        lora_rank=8,
        lora_alpha=16,
        lora_dropout=0.1,
        use_lora=True
    ):
        self.model_name = model_name
        self.use_lora = use_lora
        
        if device == "auto":
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)
            
        print(f"🔧 Initializing ProteinEmbeddingExtractor on {self.device}")
        
        self.dtype = torch.float16 if self.device.type == 'cuda' else torch.float32
        print(f"📥 Loading ESM-2 model: {model_name} with dtype: {self.dtype}")
        
        self.model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=self.dtype
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        if self.use_lora:
            print(f"✨ Applying LoRA with rank={lora_rank}, alpha={lora_alpha}, dropout={lora_dropout}")
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                bias="none",
                task_type=TaskType.FEATURE_EXTRACTION,
                target_modules=["key", "query", "value"]
            )
            self.model = get_peft_model(self.model, lora_config)
            print("   ✅ LoRA model loaded and configured. Trainable parameters:")
            self.model.print_trainable_parameters()
        else:
            print("   ⚠️ LoRA is disabled. All base model weights are frozen.")
            for param in self.model.parameters():
                param.requires_grad = False

        self.model.to(self.device)
        self.embedding_size = self.model.config.hidden_size
        print(f"   ✅ Model loaded. Embedding size from config: {self.embedding_size}")

    def get_model_and_tokenizer(self):
        """Returns the prepared model (PeftModel or base) and tokenizer."""
        return self.model, self.tokenizer

# ==============================================================================
# 3. LOAD YOUR DATASET
# ==============================================================================
print("\n" + "="*80)
print("... 3. LOADING DATA FROM Data.csv ...")

protein1_col = 'Target'
protein2_col = 'proteina'
target_col = 'Y'
pdb_col_raw = 'PDB'
subgroup_col = 'Subgroup'
source_dataset_col = 'Source Data Set' # NEW
try:
    df_raw = pd.read_csv(r"C:\Users\hs494\PPIBALM\BALM-PPI\scripts\notebooks\PPB_Affinity_Sequences_Final (version 1).csv")
    print(f"Successfully loaded Data.csv with {len(df_raw)} rows.")
    df = df_raw[[protein1_col, protein2_col, target_col, pdb_col_raw, subgroup_col,source_dataset_col]].copy()
    df.rename(columns={
        protein1_col: 'Target',
        protein2_col: 'proteina',
        target_col: 'Y',
        subgroup_col : 'Subgroup',
        source_dataset_col: 'SourceDataSet' # NEW
    }, inplace=True)
    
    print("\n--- Initial Data Check ---")
    print(df.head())
    missing_values = df.isnull().sum()
    print("\nMissing values per column:\n", missing_values)
    
    if missing_values.sum() > 0:
        print(f"\nWarning: Found {missing_values.sum()} total missing values. Dropping rows with NaNs.")
        df.dropna(subset=['Target', 'proteina', 'Y', 'PDB'], inplace=True)
        print(f"Dataset now has {len(df)} rows after cleaning.")
        
    df['Target'] = df['Target'].astype(str)
    df['proteina'] = df['proteina'].astype(str)
    df['Y'] = pd.to_numeric(df['Y'], errors='coerce')
    df['PDB'] = df['PDB'].astype(str)
    df['Subgroup'] = df['Subgroup'].astype(str)
    df['SourceDataSet'] = df['SourceDataSet'].astype(str) # NEW

    df.dropna(subset=['Y'], inplace=True)
    
    print(f"\n✅ Data is loaded and cleaned. Final dataset size: {len(df)} samples.")
    df.info()

except FileNotFoundError:
    print("❌ ERROR: 'Data.csv' not found.")
    df = pd.DataFrame(columns=['Target', 'proteina', 'Y', 'PDB'])
except KeyError as e:
    print(f"❌ ERROR: Column {e} not found in 'Data.csv'.")
    df = pd.DataFrame(columns=['Target', 'proteina', 'Y', 'PDB'])

# ==============================================================================
# 4. EXPERIMENT CONFIGURATION
# ==============================================================================
print("\n" + "="*80)
print("... 4. CONFIGURING THE EXPERIMENT ...")

CV_STRATEGY = 'SequenceSimilarity' 
N_SPLITS = 5
EPOCHS = 30
PATIENCE = 15
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
PROJECTED_SIZE = 256
DROPOUT = 0.1

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
USE_LORA_FOR_TRAINING = True

# Sequence Similarity CV Configs
SIMILARITY_THRESHOLD = 0.3
K_MER_SIZE = 3
CLUSTERING_METHOD = 'agglomerative'

RESULTS_DIR = "cv_results_lora_seqsim" # Use a unique name
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved in: {RESULTS_DIR}")

print(f"✅ Configuration set. Using CV Strategy: {CV_STRATEGY}")
print(f"   Epochs: {EPOCHS}, Patience: {PATIENCE}")
if USE_LORA_FOR_TRAINING:
    print(f"   LoRA Enabled: Rank={LORA_RANK}, Alpha={LORA_ALPHA}, Dropout={LORA_DROPOUT}")
else:
    print(f"   LoRA Disabled: Only projection head will be trained.")

# ==============================================================================
# 5. ADAPTED MODEL AND DATASET
# ==============================================================================
print("\n" + "="*80)
print("... 5. DEFINING ADAPTED MODEL AND DATASET ...")

class BALMProjectionHead(nn.Module):
    def __init__(self, embedding_size, projected_size, projected_dropout):
        super().__init__()
        self.protein_projection = nn.Linear(embedding_size, projected_size)
        self.proteina_projection = nn.Linear(embedding_size, projected_size)
        self.dropout = nn.Dropout(projected_dropout)
        self.loss_fn = nn.MSELoss()
        print(f"✅ BALMProjectionHead initialized with embedding_size: {embedding_size}")

    def forward(self, protein_embedding, proteina_embedding, labels=None):
        protein_embedding = self.dropout(protein_embedding)
        proteina_embedding = self.dropout(proteina_embedding)
        protein_projected = self.protein_projection(protein_embedding)
        proteina_projected = self.proteina_projection(proteina_embedding)
        protein_projected = F.normalize(protein_projected, p=2, dim=1)
        proteina_projected = F.normalize(proteina_projected, p=2, dim=1)
        cosine_similarity = F.cosine_similarity(protein_projected, proteina_projected)
        cosine_similarity = torch.clamp(cosine_similarity, -0.9999, 0.9999)
        output = {"cosine_similarity": cosine_similarity}
        if labels is not None:
            loss = self.loss_fn(cosine_similarity, labels)
            output["loss"] = loss
        return output

class BALMForLoRAFinetuning(nn.Module):
    def __init__(self, esm_model, esm_tokenizer, projected_size, projected_dropout, pkd_bounds):
        super().__init__()
        self.esm_model = esm_model
        self.esm_tokenizer = esm_tokenizer
        self.embedding_size = self.esm_model.config.hidden_size
        self.projection_head = BALMProjectionHead(self.embedding_size, projected_size, projected_dropout)
        self.pkd_lower, self.pkd_upper = pkd_bounds
        self.pkd_range = self.pkd_upper - self.pkd_lower
        print(f"✅ BALMForLoRAFinetuning model initialized.")

    def _get_esm_embeddings(self, sequences: List[str]):
        cls_token = self.esm_tokenizer.cls_token
        processed_seqs = [s.replace('|', f"{cls_token}{cls_token}") for s in sequences]
        
        inputs = self.esm_tokenizer(processed_seqs, return_tensors="pt", padding=True, truncation=True, max_length=1024)
        inputs = {k: v.to(self.esm_model.device) for k, v in inputs.items()}
        outputs = self.esm_model(**inputs)
        
        # Pooling logic (remains Mean Pooling, which now covers the internal CLS tokens)
        hidden_states = outputs.last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).expand(hidden_states.size()).float()
        return torch.sum(hidden_states * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

    def forward(self, batch_input):
        protein_emb = self._get_esm_embeddings(batch_input["protein_sequence"])
        proteina_emb = self._get_esm_embeddings(batch_input["proteina_sequence"])
        labels = batch_input.get("labels")
        proj_output = self.projection_head(protein_emb, proteina_emb, labels)
        return {
            "cosine_similarity": proj_output["cosine_similarity"],
            "original_pkds": batch_input["original_pkds"],
            "pdb_groups": batch_input["pdb_groups"],
            "subgroups": batch_input["subgroups"],
            "source_dataset": batch_input["source_dataset"], # NEW
            "loss": proj_output.get("loss")
        }
        if "loss" in proj_output:

            output["loss"] = proj_output["loss"]
        return output

class ProteinPairSequenceDataset(Dataset):
    def __init__(self, dataframe, pkd_bounds):
        self.data = dataframe.reset_index(drop=True)
        self.pkd_lower, self.pkd_upper = pkd_bounds
        self.pkd_range = self.pkd_upper - self.pkd_lower

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        original_pkd = float(row['Y'])
        scaled_label = ((original_pkd - self.pkd_lower) / self.pkd_range) * 2 - 1
        return {
            "protein_sequence": row["Target"],
            "proteina_sequence": row["proteina"],
            "labels": torch.tensor(scaled_label, dtype=torch.float32),
            "original_pkds": torch.tensor(original_pkd, dtype=torch.float32),
            "pdb_groups": row["PDB"],
            "subgroups": row["Subgroup"],
            "source_dataset": row["SourceDataSet"] # NEW
        }

def collate_fn_sequences(batch):
    return {
        "protein_sequence": [item['protein_sequence'] for item in batch],
        "proteina_sequence": [item['proteina_sequence'] for item in batch],
        "labels": torch.stack([item['labels'] for item in batch]),
        "original_pkds": torch.stack([item['original_pkds'] for item in batch]),
        "pdb_groups": [item['pdb_groups'] for item in batch], # NEW
        "subgroups": [item['subgroups'] for item in batch],  # NEW
        "source_dataset": [item['source_dataset'] for item in batch] # NEW
    }

# ==============================================================================
# 6. METRICS, EVALUATION & SEQUENCE SIMILARITY HELPERS (NEW)
# ==============================================================================
print("\n" + "="*80)
print("... 6. DEFINING METRICS, EVALUATION & SEQUENCE HELPERS ...")

def concordance_index(y_true, y_pred):
    try:
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        n = len(y_true)
        if n < 2: return 1.0
        concordant, discordant, tied_pred = 0, 0, 0
        for i in range(n):
            for j in range(i + 1, n):
                if y_true[i] == y_true[j]: continue
                if y_pred[i] == y_pred[j]:
                    tied_pred += 1
                    continue
                if (y_pred[i] > y_pred[j] and y_true[i] > y_true[j]) or \
                   (y_pred[i] < y_pred[j] and y_true[i] < y_true[j]):
                    concordant += 1
                else:
                    discordant += 1
        total_pairs = concordant + discordant + tied_pred
        return 1.0 if total_pairs == 0 else (concordant + 0.5 * tied_pred) / total_pairs
    except Exception:
        return np.nan

def evaluate_model(model, data_loader, device, pkd_bounds):
    model.eval()
    all_labels, all_preds_pkd, all_pdb_groups, all_subgroups, all_sources = [], [], [], [], [] 
    pkd_lower, pkd_range = pkd_bounds[0], pkd_bounds[1] - pkd_bounds[0]
    
    with torch.no_grad():
        for batch in data_loader:
            batch_gpu = {k: v for k, v in batch.items()}
            for key in ["labels", "original_pkds"]:
                if key in batch_gpu: batch_gpu[key] = batch_gpu[key].to(device)
            outputs = model(batch_gpu)
            cosine_sim = outputs['cosine_similarity'].cpu()
            preds_pkd = ((cosine_sim + 1) / 2) * pkd_range + pkd_lower
            all_labels.extend(batch['original_pkds'].numpy())
            all_preds_pkd.extend(preds_pkd.numpy())
            all_pdb_groups.extend(batch['pdb_groups']) 
            all_subgroups.extend(batch['subgroups'])   
            all_sources.extend(batch['source_dataset']) # NEW

    labels, preds = np.array(all_labels), np.array(all_preds_pkd)
    pdb_groups_arr = np.array(all_pdb_groups)
    subgroups_arr = np.array(all_subgroups)
    sources_arr = np.array(all_sources) # NEW
    
    mask = ~np.isnan(labels) & ~np.isnan(preds)
    labels, preds = labels[mask], preds[mask]
    
    if len(labels) < 2:
        return {"rmse": np.nan, "pearson": np.nan, "spearman": np.nan, "ci": np.nan}, labels, preds, pdb_groups_arr, subgroups_arr, sources_arr
    return {
        'rmse': np.sqrt(mean_squared_error(labels, preds)),
        'pearson': pearsonr(labels, preds)[0],
        'spearman': spearmanr(labels, preds)[0],
        'ci': concordance_index(labels, preds)
    }, labels, preds, pdb_groups_arr, subgroups_arr, sources_arr

def plot_regression(y_true, y_pred, metrics, title, filename=None):
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.6, edgecolor='k', s=50)
    min_val, max_val = min(np.min(y_true), np.min(y_pred)), max(np.max(y_true), np.max(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Identity Line (y=x)')
    metrics_text = f"RMSE: {metrics['rmse']:.4f}\nPearson: {metrics['pearson']:.4f}\nSpearman: {metrics['spearman']:.4f}\nCI: {metrics['ci']:.4f}"
    plt.text(0.05, 0.95, metrics_text, transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5))
    plt.title(title, fontsize=16)
    plt.xlabel("Actual pKd", fontsize=14)
    plt.ylabel("Predicted pKd", fontsize=14)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if filename:                       # NEW
        plt.savefig(filename)          # NEW
        print(f"Plot saved to {filename}") # NEW
    plt.show()

def compute_kmer_similarity_matrix(sequences, k=3):
    """
    Compute pairwise sequence similarity matrix using k-mer based Jaccard similarity.
    This is a biologically relevant method for sequence similarity computation.
    """
    print(f"Computing {k}-mer Jaccard similarity matrix for {len(sequences)} sequences...")
    
    def get_kmers(sequence, k_size):
        """Extract k-mers from a sequence."""
        if len(sequence) < k_size:
            return {sequence}
        return {sequence[i:i+k_size] for i in range(len(sequence) - k_size + 1)}
    
    def jaccard_similarity(seq1, seq2):
        """Compute Jaccard similarity between two sequences based on k-mers."""
        kmers1 = get_kmers(seq1, k)
        kmers2 = get_kmers(seq2, k)
        if not kmers1 or not kmers2:
            return 0.0
        intersection = len(kmers1 & kmers2)
        union = len(kmers1 | kmers2)
        return intersection / union if union > 0 else 0.0
    
    n_seqs = len(sequences)
    similarity_matrix = np.zeros((n_seqs, n_seqs))
    
    # Compute similarity matrix with progress bar
    for i in tqdm(range(n_seqs), desc="Computing similarities"):
        for j in range(i, n_seqs):
            if i == j:
                similarity_matrix[i, j] = 1.0
            else:
                sim = jaccard_similarity(sequences[i], sequences[j])
                similarity_matrix[i, j] = sim
                similarity_matrix[j, i] = sim
    
    avg_similarity = np.mean(similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)])
    print(f"Average pairwise similarity: {avg_similarity:.3f}")
    
    return similarity_matrix

def cluster_sequences_by_similarity(similarity_matrix, threshold=0.3, method='agglomerative'):
    """
    Cluster sequences based on similarity matrix using specified clustering method.
    Uses distance threshold derived from similarity threshold.
    MODIFIED: Uses scipy.cluster.hierarchy for deterministic clustering.
    """
    print(f"Clustering sequences using {method} method with similarity threshold {threshold}")
    
    # Convert similarity to distance matrix
    distance_matrix = 1.0 - similarity_matrix
    
    try:
        if method == 'agglomerative':
            from scipy.spatial.distance import squareform
            from scipy.cluster.hierarchy import linkage, fcluster
            
            # Use distance threshold - sequences with similarity > threshold will be in same cluster
            distance_threshold = 1.0 - threshold
            
            print(f"Using distance threshold: {distance_threshold}")
            print(f"Distance matrix stats - Min: {distance_matrix.min():.3f}, Max: {distance_matrix.max():.3f}, Mean: {distance_matrix.mean():.3f}")
            
            # Convert to condensed distance matrix (required by scipy.linkage)
            # squareform converts square distance matrix to condensed vector
            condensed_dist = squareform(distance_matrix, checks=False)
            
            # Perform hierarchical clustering (DETERMINISTIC)
            # method='average' corresponds to UPGMA (average linkage)
            Z = linkage(condensed_dist, method='average')
            
            # Cut dendrogram at distance threshold to form flat clusters
            # criterion='distance' cuts at specified height
            clusters = fcluster(Z, t=distance_threshold, criterion='distance')
            
            # Convert to 0-indexed (fcluster returns 1-indexed)
            clusters = clusters - 1
            
            print(f"Hierarchical clustering completed successfully (deterministic)")
            
        else:
            raise ValueError(f"Unsupported clustering method: {method}")
        
        if clusters is None:
            raise ValueError("Clustering failed - returned None")
        
        n_clusters = len(set(clusters))
        print(f"Created {n_clusters} clusters")
        
        cluster_sizes = [np.sum(clusters == i) for i in set(clusters)]
        print(f"Cluster size distribution - Min: {min(cluster_sizes)}, Max: {max(cluster_sizes)}, Mean: {np.mean(cluster_sizes):.1f}")
        
        return clusters
        
    except Exception as e:
        print(f"❌ Clustering failed with error: {e}")
        print("🔄 Falling back to individual clusters (each sequence in its own cluster)")
        
        n_sequences = similarity_matrix.shape[0]
        fallback_clusters = np.arange(n_sequences)
        
        print(f"Created {n_sequences} individual clusters (fallback)")
        return fallback_clusters

# ==============================================================================
# 7. CROSS-VALIDATION TRAINING (MODIFIED FOR 80/20 SPLIT + TRAIN LOSS PATIENCE)
# ==============================================================================
print("\n" + "="*80)
print(f"... 7. RUNNING '{CV_STRATEGY}' STRATEGY WITH LoRA (80/20 Split with Training Loss Patience) ...")

def run_sequence_similarity_cv_with_lora():
    if df.empty:
        print("⚠️ Skipping training due to empty dataframe.")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    PKD_LOWER_BOUND, PKD_UPPER_BOUND = df['Y'].min(), df['Y'].max()
    PKD_BOUNDS = (PKD_LOWER_BOUND, PKD_UPPER_BOUND)
    print(f"Global pKd bounds for scaling: [{PKD_LOWER_BOUND:.2f}, {PKD_UPPER_BOUND:.2f}]")

    # --- Steps 1-4: Data Preparation and Clustering ---
    print("\n🧬 Step 1: Extracting unique protein sequences...")
    sequence_columns = ['Target', 'proteina']
    all_sequences = set()
    sequence_to_records = {}
    for idx, row in df.iterrows():
        for col in sequence_columns:
            seq = str(row[col]).strip()
            if seq:
                all_sequences.add(seq)
                if seq not in sequence_to_records:
                    sequence_to_records[seq] = []
                sequence_to_records[seq].append(idx)
    unique_sequences = sorted(list(all_sequences))
    print(f"Found {len(unique_sequences)} unique protein sequences.")

    print("\n🧬 Step 2: Computing sequence similarity matrix...")
    similarity_matrix = compute_kmer_similarity_matrix(unique_sequences, k=K_MER_SIZE)

    print("\n🧬 Step 3: Clustering sequences...")
    clusters = cluster_sequences_by_similarity(similarity_matrix, threshold=SIMILARITY_THRESHOLD, method=CLUSTERING_METHOD)
    
    cluster_to_records = {}
    for seq_idx, cluster_id in enumerate(clusters):
        seq = unique_sequences[seq_idx]
        if cluster_id not in cluster_to_records:
            cluster_to_records[cluster_id] = []
        cluster_to_records[cluster_id].extend(sequence_to_records[seq])
    
    for cluster_id in cluster_to_records:
        cluster_to_records[cluster_id] = list(set(cluster_to_records[cluster_id]))
    # --- End of omitted code ---



    # --- Step 5: Perform Cluster-based Cross-Validation ---
    print(f"\n🚀 Starting {N_SPLITS}-Fold **{CV_STRATEGY}** Cross-Validation with LoRA fine-tuning...")

    # NEW: Display cluster count and fold split preview
    print(f"\nCreated {len(set(clusters))} clusters")
    print(f"   Step 4: Creating {N_SPLITS}-fold splits...")

    # Pre-compute and display split sizes
    cluster_ids = list(cluster_to_records.keys())
    kf_preview = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

    for preview_fold, (train_cluster_idx, test_cluster_idx) in enumerate(kf_preview.split(cluster_ids)):
        preview_fold_num = preview_fold + 1
        
        train_clusters_preview = set([cluster_ids[i] for i in train_cluster_idx])
        test_clusters_preview = set([cluster_ids[i] for i in test_cluster_idx])
        
        sequence_to_group_preview = {}
        for seq_idx, cluster_id in enumerate(clusters):
            sequence = unique_sequences[seq_idx]
            if cluster_id in train_clusters_preview:
                sequence_to_group_preview[sequence] = 'train'
            elif cluster_id in test_clusters_preview:
                sequence_to_group_preview[sequence] = 'test'
        
        train_count, test_count = 0, 0
        for idx, row in df.iterrows():
            seq_target = str(row['Target']).strip()
            seq_proteina = str(row['proteina']).strip()
            
            cluster_target_id = sequence_to_group_preview.get(seq_target)
            cluster_proteina_id = sequence_to_group_preview.get(seq_proteina)
            
            if cluster_target_id == 'test' or cluster_proteina_id == 'test':
                test_count += 1
            else:
                train_count += 1
        
        print(f"   Fold {preview_fold_num}: Train={train_count}, Test={test_count}")

    print(f"\n🔄 Beginning actual training across {N_SPLITS} folds...")

    
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    
    fold_results, all_folds_y_true, all_folds_y_pred = [], [], []
    
    for fold, (train_cluster_idx, test_cluster_idx) in enumerate(kf.split(cluster_ids)):
        fold_num = fold + 1
        print(f"\n===== STRATEGY: {CV_STRATEGY} | FOLD {fold_num}/{N_SPLITS} =====")
        
        # --- CRITICAL FIX: RE-INITIALIZE MODEL HERE ---
        # This ensures the LoRA weights are reset for every fold.
        print(f"🔄 Initializing fresh ESM-2 Model and LoRA Adapters for Fold {fold_num}...")
        extractor = ProteinEmbeddingExtractor(
            device=device, use_lora=USE_LORA_FOR_TRAINING, lora_rank=LORA_RANK,
            lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT
        )
        esm_model, esm_tokenizer = extractor.get_model_and_tokenizer()
        # ----------------------------------------------

        fold_model_path = os.path.join(RESULTS_DIR, f"best_model_fold_{fold_num}.pth")
        
        train_clusters = [cluster_ids[i] for i in train_cluster_idx]
        test_clusters = [cluster_ids[i] for i in test_cluster_idx]

        # Create a map from each unique sequence to its group ('train' or 'test')
        sequence_to_group = {}
        train_cluster_set = set(train_clusters)
        test_cluster_set = set(test_clusters)

        for seq_idx, cluster_id in enumerate(clusters):
            sequence = unique_sequences[seq_idx]
            if cluster_id in train_cluster_set:
                sequence_to_group[sequence] = 'train'
            elif cluster_id in test_cluster_set:
                sequence_to_group[sequence] = 'test'

        # Iterate through the main dataframe and assign rows
        train_indices, test_indices = [], []
        
        for idx, row in df.iterrows():
            seq_target = str(row['Target']).strip()
            seq_proteina = str(row['proteina']).strip()

            cluster_target_id = sequence_to_group.get(seq_target)
            cluster_proteina_id = sequence_to_group.get(seq_proteina)

            # Strict Leakage Rule: If EITHER protein is in a test cluster, it goes to test.
            if cluster_target_id == 'test' or cluster_proteina_id == 'test':
                test_indices.append(idx)
            else:
                train_indices.append(idx)

        # Create dataframes
        train_df = df.loc[train_indices]
        test_df = df.loc[test_indices]
        
        assert len(set(train_indices).intersection(set(test_indices))) == 0, "Overlap found between train and test sets!"

        print(f"  Total samples allocated: {len(train_df) + len(test_df)}")
        print(f"  Train: {len(train_df)} samples")
        print(f"  Test: {len(test_df)} samples")

        # --- Use GLOBAL Scaling Bounds (for consistent comparison across folds) ---
        print(f"  Using Global Scaling Bounds: {PKD_BOUNDS}")

        # --- DataLoaders ---
        train_dataset = ProteinPairSequenceDataset(train_df, PKD_BOUNDS)
        test_dataset = ProteinPairSequenceDataset(test_df, PKD_BOUNDS)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_sequences)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn_sequences)

        # --- Model, Optimizer ---
        model = BALMForLoRAFinetuning(
            esm_model=esm_model, esm_tokenizer=esm_tokenizer,
            projected_size=PROJECTED_SIZE, projected_dropout=DROPOUT,
            pkd_bounds=PKD_BOUNDS
        ).to(device)
        
        optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
        print(f"   Optimizing {sum(p.numel() for p in model.parameters() if p.requires_grad)} trainable parameters.")

        # --- Training Loop ---
        best_val_metric = float('inf')
        patience_counter = 0

        for epoch in range(EPOCHS):
            model.train()
            total_train_loss = 0.0
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
            
            for batch in progress_bar:
                optimizer.zero_grad()
                batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
                outputs = model(batch_gpu)
                loss = outputs['loss']
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()
                progress_bar.set_postfix({'batch_loss': f"{loss.item():.4f}"})
            
            avg_train_loss = total_train_loss / len(train_loader)
            # Evaluate on test set
            val_metrics, _, _, _, _, _ = evaluate_model(model, test_loader, device, PKD_BOUNDS)
            current_val_metric = val_metrics['rmse']
            print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Test RMSE: {current_val_metric:.4f} | Test Pearson: {val_metrics['pearson']:.4f}")
            
            # Save best model based on Validation RMSE (standard practice)
            if current_val_metric < best_val_metric:
                best_val_metric = current_val_metric
                patience_counter = 0
                torch.save(model.state_dict(), fold_model_path)
                print(f"   -> New best model saved with Val RMSE: {best_val_metric:.4f}")
            else:
                patience_counter += 1
            
            if patience_counter >= PATIENCE:
                print(f"⏳ Early stopping triggered at epoch {epoch+1}. Val RMSE has not improved for {PATIENCE} epochs.")
                break
        
        print(f"Fold {fold_num} training complete.")        
        # --- Final Evaluation on Test Set ---
        print("Loading best model (based on lowest train loss) for final TEST evaluation...")
        model.load_state_dict(torch.load(fold_model_path)) 
        
        final_test_metrics, y_true_fold, y_pred_fold, pdb_groups_fold, subgroups_fold, sources_fold = evaluate_model(model, test_loader, device, PKD_BOUNDS)
        print(f"Fold {fold_num} Final Test Metrics:")
        for key, value in final_test_metrics.items():
            print(f"  - {key.upper()}: {value:.4f}")

        fold_results.append(final_test_metrics)
        all_folds_y_true.extend(y_true_fold)
        all_folds_y_pred.extend(y_pred_fold)

        # --- Save Predictions for the Fold ---
        # MODIFIED: Create a more detailed DataFrame
        pred_df = pd.DataFrame({
            'label': y_true_fold, 
            'prediction': y_pred_fold, 
            'residual': y_true_fold - y_pred_fold,
            'abs_residual': np.abs(y_true_fold - y_pred_fold),
            'PDB': pdb_groups_fold, 
            'Subgroup': subgroups_fold,
            'Source Data Set': sources_fold  # Preserved column
        })
        # MODIFIED: Save to the RESULTS_DIR
        pred_filename = os.path.join(RESULTS_DIR, f"fold_{fold_num}_predictions.csv")
        pred_df.to_csv(pred_filename, index=False)
        print(f"   💾 Predictions for fold {fold_num} saved to '{pred_filename}'")


    # --- Aggregate and Display Final Results  ---
    print(f"\n📈 Generating combined regression plot for {CV_STRATEGY}")
    combined_y_true = np.array(all_folds_y_true)
    combined_y_pred = np.array(all_folds_y_pred)

    overall_metrics = {
        'rmse': np.sqrt(mean_squared_error(combined_y_true, combined_y_pred)),
        'pearson': pearsonr(combined_y_true, combined_y_pred)[0],
        'spearman': spearmanr(combined_y_true, combined_y_pred)[0],
        'ci': concordance_index(combined_y_true, combined_y_pred)
    }
    
    plot_regression(
    combined_y_true,
    combined_y_pred,
    overall_metrics,
    title=f"Overall Regression for {CV_STRATEGY} ({N_SPLITS}-Fold CV) with LoRA Fine-tuning",
    filename=os.path.join(RESULTS_DIR, "overall_regression_plot.png")
    )

    print(f"\n✅ Final Results Summary for {CV_STRATEGY} with LoRA Fine-tuning:")
    results_df = pd.DataFrame(fold_results)
    summary_df = pd.DataFrame({'Mean': results_df.mean(), 'Std Dev': results_df.std()})
    print(summary_df)

    # NEW: Save the summary metrics to a CSV file
    summary_filename = os.path.join(RESULTS_DIR, "cv_summary_metrics.csv")
    summary_df.to_csv(summary_filename)
    print(f"Overall CV summary metrics saved to {summary_filename}")
    print(f"Fold model checkpoints and prediction files are saved in the '{RESULTS_DIR}' directory.")

# --- Call the new CV function ---
run_sequence_similarity_cv_with_lora()


# ==============================================================================
# 8. FINAL NOTES AND RECOMMENDATIONS
# ==============================================================================
print("\n" + "="*80)
print("... 8. FINAL NOTES AND RECOMMENDATIONS ...")
print("""
🎉 CONGRATULATIONS! You've successfully implemented BALM with on-the-fly ESM-2 embeddings
and **Parameter-Efficient Fine-Tuning (PEFT) using LoRA**!

📊 KEY ADVANTAGES OF THIS APPROACH:
• **Efficiency:** Fine-tunes only a tiny fraction of the ESM-2 model parameters (<1%).
• **Speed & Memory:** Significantly faster and less memory-intensive than full fine-tuning.
• **Performance:** Adapts the powerful ESM-2 model to your specific data, often leading to better performance.
• **No Static Cache:** By computing embeddings dynamically, the model fully leverages the fine-tuned LoRA adapters during training.

🚀 PERFORMANCE OPTIMIZATION TIPS:
1.  **LoRA Hyperparameters:**
    *   `lora_rank (r)`: Controls the capacity of the adapters. Higher `r` means more parameters. Typical values are 8-64.
    *   `lora_alpha`: Acts as a scaling factor. A common heuristic is to set `lora_alpha` to be twice the `lora_rank`. [7]
    *   `target_modules`: Experiment with adding other modules like `dense` to the target list in the `LoraConfig`. [1]
2.  **Learning Rate:** LoRA can sometimes handle slightly higher learning rates than full fine-tuning. Experiment with values from 5e-5 to 5e-4.
3.  **Batch Size & Gradient Accumulation:** If your GPU memory is a constraint, you can simulate a larger batch size by using gradient accumulation.

💾 SAVING AND LOADING THE FINE-TUNED MODEL:
• After training, you can save the trained LoRA adapters separately from the base model. This is very efficient for storage. [3]
• To save: `model.esm_model.save_pretrained("path/to/my_lora_adapters")` [3]
• To load: First, load the base ESM-2 model. Then, use `PeftModel.from_pretrained(base_model, "path/to/my_lora_adapters")`.

🎯 NEXT STEPS:
1.  **Hyperparameter Search:** Systematically tune LoRA parameters (`r`, `lora_alpha`) and the learning rate for optimal performance.
2.  **Experiment with Larger ESM-2 Models:** LoRA makes fine-tuning even the largest ESM-2 models (e.g., 3B parameters) feasible on a single high-end GPU.
3.  **Advanced PEFT Techniques:** Explore other PEFT methods like IA3 or full parameter-efficient fine-tuning for comparison.
""")
print("\n✅ LoRA PEFT implementation complete! Happy parameter-efficient protein modeling! 🧬🤖")


... 0. PEFT LIBRARY INITIALIZED ...
    Ready for Parameter-Efficient Fine-Tuning with LoRA!
🧬 BALM Adaptation for ESM-2 with LoRA Fine-Tuning
⚡ Fast Training with 5-Fold Cross-Validation
🔒 Setting up reproducibility with seed 42
✅ Reproducibility setup complete

... 3. LOADING DATA FROM Data.csv ...
Successfully loaded Data.csv with 12062 rows.

--- Initial Data Check ---
                                              Target  \
0  PKFTKCRSPERETFSCHWTLGPIQLFYTRRNTQEWTQEWKECPDYV...   
1  QDNSRYTHFLTQHYDAKPQGRDDRYCESIMRRRGLTSPCKDINTFI...   
2  KSFPEVVGKTVDQAREYFTLHYPQYDVYFLPEGSPVTLDLRYNRVR...   
3  TNTVAAYNLTWKSTNFKTILEWEPKPVNQVYTVQISTKSGDWKSKC...   
4  PIVQNLQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEG...   

                                            proteina          Y   PDB  \
0  FPTIPLSRLFDNAMLRAHRLHQLAFDTYQEFEEAYIPKEQKYSFLQ...   9.045757  1A22   
1  SLDIQSLDIQCEELSDARWAELLPLLQQCQVVRLDDCGLTEARCKD...  15.301030  1A4Y   
2  CGVPAIQPVLSGLIVNGEEAVPGSWPWQVSLQDKTGFHFCGGSLIN...  11.826814  1A

Computing similarities: 100%|██████████| 11598/11598 [1:00:04<00:00,  3.22it/s] 


Average pairwise similarity: 0.028

🧬 Step 3: Clustering sequences...
Clustering sequences using agglomerative method with similarity threshold 0.3
Using distance threshold: 0.7
Distance matrix stats - Min: 0.000, Max: 1.000, Mean: 0.972
Hierarchical clustering completed successfully (deterministic)
Created 2678 clusters
Cluster size distribution - Min: 1, Max: 1060, Mean: 4.3

🚀 Starting 5-Fold **SequenceSimilarity** Cross-Validation with LoRA fine-tuning...

Created 2678 clusters
   Step 4: Creating 5-fold splits...
   Fold 1: Train=6516, Test=5503
   Fold 2: Train=7503, Test=4516
   Fold 3: Train=7340, Test=4679
   Fold 4: Train=7331, Test=4688
   Fold 5: Train=9397, Test=2622

🔄 Beginning actual training across 5 folds...

===== STRATEGY: SequenceSimilarity | FOLD 1/5 =====
🔄 Initializing fresh ESM-2 Model and LoRA Adapters for Fold 1...
🔧 Initializing ProteinEmbeddingExtractor on cuda
📥 Loading ESM-2 model: facebook/esm2_t33_650M_UR50D with dtype: torch.float16


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✨ Applying LoRA with rank=8, alpha=16, dropout=0.1
   ✅ LoRA model loaded and configured. Trainable parameters:
trainable params: 2,027,520 || all params: 654,381,461 || trainable%: 0.3098
   ✅ Model loaded. Embedding size from config: 1280
  Total samples allocated: 12019
  Train: 6516 samples
  Test: 5503 samples
  Using Global Scaling Bounds: (1.3187587626244128, 15.69897000433602)
✅ BALMProjectionHead initialized with embedding_size: 1280
✅ BALMForLoRAFinetuning model initialized.
   Optimizing 2683392 trainable parameters.


...

Epoch 16/30 | Train Loss: 0.0112 | Test RMSE: 1.6025 | Test Pearson: 0.5816


Epoch 17/30 | Train Loss: 0.0107 | Test RMSE: 1.6086 | Test Pearson: 0.5830
⏳ Early stopping triggered at epoch 17. Val RMSE has not improved for 15 epochs.
Fold 5 training complete.
Loading best model (based on lowest train loss) for final TEST evaluation...


C:\Users\hs494\AppData\Local\Temp\ipykernel_46900\1832587606.py:744: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(fold_model_path))


Fold 5 Final Test Metrics:
  - RMSE: 1.5507
  - PEARSON: 0.6019
  - SPEARMAN: 0.5709
  - CI: 0.7011
   💾 Predictions for fold 5 saved to 'cv_results_lora_seqsim\fold_5_predictions.csv'

📈 Generating combined regression plot for SequenceSimilarity
Plot saved to cv_results_lora_seqsim\overall_regression_plot.png

✅ Final Results Summary for SequenceSimilarity with LoRA Fine-tuning:
              Mean   Std Dev
rmse      1.672272  0.195301
pearson   0.612109  0.086380
spearman  0.601683  0.086626
ci        0.713320  0.034457
Overall CV summary metrics saved to cv_results_lora_seqsim\cv_summary_metrics.csv
Fold model checkpoints and prediction files are saved in the 'cv_results_lora_seqsim' directory.

... 8. FINAL NOTES AND RECOMMENDATIONS ...

🎉 CONGRATULATIONS! You've successfully implemented BALM with on-the-fly ESM-2 embeddings
and **Parameter-Efficient Fine-Tuning (PEFT) using LoRA**!

📊 KEY ADVANTAGES OF THIS APPROACH:
• **Efficiency:** Fine-tunes only a tiny fraction of the ESM-2 m

C:\Users\hs494\AppData\Local\Temp\ipykernel_46900\1832587606.py:423: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
